# Information Retrieval: From Theory to Implementation
**Kyle Hillhouse** | Kennesaw State University

This notebook implements core Information Retrieval models from scratch:
- Boolean Retrieval
- Unigram Language Model
- Bigram Language Model
- TF-IDF Ranking
- BM25 Ranking
- Query Likelihood Retrieval Model


## Setup & Sample Corpus
We'll use a small corpus of documents about technology topics to demonstrate each model.

In [ ]:
import math
import re
from collections import Counter, defaultdict

# Sample corpus - each entry is a (doc_id, text) pair
corpus = [
    ("doc1", "information retrieval systems help users find relevant documents"),
    ("doc2", "machine learning models improve information extraction from text"),
    ("doc3", "image retrieval uses visual features to search photo databases"),
    ("doc4", "natural language processing enables information retrieval in search engines"),
    ("doc5", "deep learning improves image recognition and retrieval accuracy"),
    ("doc6", "text mining and information extraction are key tasks in data science"),
]

def tokenize(text):
    """Lowercase and split text into tokens."""
    return re.findall(r'\b\w+\b', text.lower())

# Build tokenized corpus
tokenized_corpus = [(doc_id, tokenize(text)) for doc_id, text in corpus]

print("Corpus loaded:")
for doc_id, tokens in tokenized_corpus:
    print(f"  {doc_id}: {tokens}")

---
## 1. Boolean Retrieval
The simplest IR model. Documents either match a query or they don't — no ranking, just yes/no.

We build an **inverted index**: a dictionary mapping each word to the set of documents containing it.

In [ ]:
class BooleanRetrieval:
    def __init__(self, tokenized_corpus):
        self.index = defaultdict(set)
        for doc_id, tokens in tokenized_corpus:
            for token in tokens:
                self.index[token].add(doc_id)

    def search(self, query, operator="AND"):
        """
        Search using AND or OR logic.
        AND: doc must contain ALL query terms
        OR:  doc must contain AT LEAST ONE query term
        """
        terms = tokenize(query)
        if not terms:
            return set()
        
        result = self.index.get(terms[0], set()).copy()
        for term in terms[1:]:
            postings = self.index.get(term, set())
            if operator == "AND":
                result &= postings
            else:
                result |= postings
        return result

br = BooleanRetrieval(tokenized_corpus)

print("AND search for 'information retrieval':")
print(" ", br.search("information retrieval", "AND"))

print("\nOR search for 'image learning':")
print(" ", br.search("image learning", "OR"))

---
## 2. Unigram Language Model
Each document is treated as a probability distribution over words (bag of words — word order doesn't matter).

**P(word) = count(word in corpus) / total tokens in corpus**

**P(query) = P(word1) × P(word2) × ... × P(wordN)**

In [ ]:
class UnigramModel:
    def __init__(self, tokenized_corpus):
        # Combine all tokens across all documents
        all_tokens = [token for _, tokens in tokenized_corpus for token in tokens]
        self.total_tokens = len(all_tokens)
        self.counts = Counter(all_tokens)

    def probability(self, word):
        """P(word) = count(word) / total tokens"""
        return self.counts.get(word, 0) / self.total_tokens

    def query_probability(self, query):
        """P(query) = product of P(each word) — assumes independence"""
        terms = tokenize(query)
        prob = 1.0
        for term in terms:
            p = self.probability(term)
            print(f"  P({term}) = {self.counts.get(term, 0)}/{self.total_tokens} = {p:.4f}")
            prob *= p
        return prob

unigram = UnigramModel(tokenized_corpus)

print(f"Total tokens in corpus: {unigram.total_tokens}\n")

print("P('information retrieval'):")
p = unigram.query_probability("information retrieval")
print(f"  => P('information retrieval') = {p:.6f}")

---
## 3. Bigram Language Model
Unlike unigrams, bigrams capture word order by modeling the probability of a word **given the previous word**.

**P(w2 | w1) = count(w1 w2) / count(w1)**

**P("w1 w2") = P(w1) × P(w2 | w1)**

In [ ]:
class BigramModel:
    def __init__(self, tokenized_corpus):
        all_tokens = [token for _, tokens in tokenized_corpus for token in tokens]
        self.total_tokens = len(all_tokens)
        self.unigram_counts = Counter(all_tokens)
        
        # Count bigrams (consecutive word pairs)
        self.bigram_counts = Counter()
        for _, tokens in tokenized_corpus:
            for i in range(len(tokens) - 1):
                self.bigram_counts[(tokens[i], tokens[i+1])] += 1

    def p_unigram(self, word):
        return self.unigram_counts.get(word, 0) / self.total_tokens

    def p_conditional(self, word, given):
        """P(word | given) = count(given, word) / count(given)"""
        bigram_count = self.bigram_counts.get((given, word), 0)
        given_count = self.unigram_counts.get(given, 0)
        if given_count == 0:
            return 0
        return bigram_count / given_count

    def query_probability(self, query):
        """P(query) = P(w1) * P(w2|w1) * P(w3|w2) ..."""
        terms = tokenize(query)
        if not terms:
            return 0
        
        prob = self.p_unigram(terms[0])
        print(f"  P({terms[0]}) = {prob:.4f}")
        
        for i in range(1, len(terms)):
            p_cond = self.p_conditional(terms[i], terms[i-1])
            print(f"  P({terms[i]} | {terms[i-1]}) = "
                  f"{self.bigram_counts.get((terms[i-1], terms[i]), 0)}/"
                  f"{self.unigram_counts.get(terms[i-1], 0)} = {p_cond:.4f}")
            prob *= p_cond
        return prob

bigram = BigramModel(tokenized_corpus)

print("P('information retrieval') using Bigram Model:")
p = bigram.query_probability("information retrieval")
print(f"  => P('information retrieval') = {p:.6f}")

---
## 4. TF-IDF Ranking
Ranks documents by how relevant they are to a query.

- **TF (Term Frequency):** How often does the term appear in this document?
- **IDF (Inverse Document Frequency):** How rare is this term across all documents? Rare terms are more informative.

**TF-IDF(t, d) = TF(t, d) × IDF(t)**

**IDF(t) = log(N / df(t))** where N = total docs, df(t) = docs containing term t

In [ ]:
class TFIDF:
    def __init__(self, tokenized_corpus):
        self.corpus = tokenized_corpus
        self.N = len(tokenized_corpus)
        
        # Document frequency: how many docs contain each term
        self.df = defaultdict(int)
        for _, tokens in tokenized_corpus:
            for term in set(tokens):  # set() so we count each doc only once
                self.df[term] += 1

    def tf(self, term, tokens):
        """Raw term frequency in a document."""
        return tokens.count(term)

    def idf(self, term):
        """log(N / df) — terms in fewer docs get higher IDF."""
        df = self.df.get(term, 0)
        if df == 0:
            return 0
        return math.log(self.N / df)

    def score(self, query, doc_tokens):
        """Sum TF-IDF scores for all query terms in a document."""
        terms = tokenize(query)
        return sum(self.tf(t, doc_tokens) * self.idf(t) for t in terms)

    def rank(self, query):
        """Return documents sorted by TF-IDF score."""
        scores = [
            (doc_id, self.score(query, tokens))
            for doc_id, tokens in self.corpus
        ]
        return sorted(scores, key=lambda x: x[1], reverse=True)

tfidf = TFIDF(tokenized_corpus)

query = "information retrieval"
print(f"TF-IDF Rankings for query: '{query}'\n")
for doc_id, score in tfidf.rank(query):
    # Find original text for display
    text = next(t for d, t in corpus if d == doc_id)
    print(f"  {doc_id} (score: {score:.4f}): {text}")

---
## 5. BM25 Ranking
BM25 is the industry standard improvement over TF-IDF used in systems like Elasticsearch.

It adds two key controls:
- **k1**: Controls TF saturation — repeated words eventually stop adding score
- **b**: Controls document length normalization — longer docs aren't unfairly boosted

**BM25(t, d) = IDF(t) × [TF × (k1 + 1)] / [TF + k1 × (1 - b + b × |d| / avgdl)]**

In [ ]:
class BM25:
    def __init__(self, tokenized_corpus, k1=1.5, b=0.75):
        self.corpus = tokenized_corpus
        self.k1 = k1
        self.b = b
        self.N = len(tokenized_corpus)
        
        self.doc_lengths = {doc_id: len(tokens) for doc_id, tokens in tokenized_corpus}
        self.avgdl = sum(self.doc_lengths.values()) / self.N
        
        # Document frequency
        self.df = defaultdict(int)
        for _, tokens in tokenized_corpus:
            for term in set(tokens):
                self.df[term] += 1

    def idf(self, term):
        """BM25 IDF with smoothing."""
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_id, doc_tokens):
        """BM25 score for a query against a document."""
        terms = tokenize(query)
        dl = self.doc_lengths[doc_id]
        total = 0
        for term in terms:
            tf = doc_tokens.count(term)
            idf = self.idf(term)
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            total += idf * (numerator / denominator)
        return total

    def rank(self, query):
        """Return documents sorted by BM25 score."""
        scores = [
            (doc_id, self.score(query, doc_id, tokens))
            for doc_id, tokens in self.corpus
        ]
        return sorted(scores, key=lambda x: x[1], reverse=True)

bm25 = BM25(tokenized_corpus)

query = "information retrieval"
print(f"BM25 Rankings for query: '{query}'\n")
for doc_id, score in bm25.rank(query):
    text = next(t for d, t in corpus if d == doc_id)
    print(f"  {doc_id} (score: {score:.4f}): {text}")

---
## 6. Query Likelihood Retrieval Model
Instead of asking "how relevant is this document to my query?", we ask:
> **"How likely is it that this document's language model would generate this query?"**

We score each document by P(query | document_model).

**Laplace smoothing** is applied to avoid zero probabilities for unseen words:

**P(t | d) = (tf(t,d) + 1) / (|d| + |V|)**

In [ ]:
class QueryLikelihood:
    def __init__(self, tokenized_corpus):
        self.corpus = tokenized_corpus
        # Build vocabulary
        self.vocab = set(token for _, tokens in tokenized_corpus for token in tokens)
        self.V = len(self.vocab)

    def p_term_given_doc(self, term, doc_tokens):
        """
        P(term | doc) with Laplace smoothing.
        Smoothing prevents zero probabilities for unseen terms.
        """
        tf = doc_tokens.count(term)
        return (tf + 1) / (len(doc_tokens) + self.V)

    def score(self, query, doc_tokens):
        """Log probability of query given document model (sum of logs to avoid underflow)."""
        terms = tokenize(query)
        return sum(math.log(self.p_term_given_doc(t, doc_tokens)) for t in terms)

    def rank(self, query):
        """Return documents sorted by query likelihood score."""
        scores = [
            (doc_id, self.score(query, tokens))
            for doc_id, tokens in self.corpus
        ]
        return sorted(scores, key=lambda x: x[1], reverse=True)

ql = QueryLikelihood(tokenized_corpus)

query = "information retrieval"
print(f"Query Likelihood Rankings for query: '{query}'\n")
for doc_id, score in ql.rank(query):
    text = next(t for d, t in corpus if d == doc_id)
    print(f"  {doc_id} (log-score: {score:.4f}): {text}")

---
## 7. Model Comparison
Let's compare how TF-IDF, BM25, and Query Likelihood rank the same query side by side.

In [ ]:
def compare_models(query):
    print(f"Query: '{query}'")
    print("-" * 60)
    
    tfidf_ranks = {doc: i+1 for i, (doc, _) in enumerate(tfidf.rank(query))}
    bm25_ranks  = {doc: i+1 for i, (doc, _) in enumerate(bm25.rank(query))}
    ql_ranks    = {doc: i+1 for i, (doc, _) in enumerate(ql.rank(query))}
    
    print(f"{'Doc':<8} {'TF-IDF':>8} {'BM25':>8} {'QueryLM':>10}  Text")
    print("-" * 80)
    for doc_id, text in corpus:
        print(f"{doc_id:<8} {tfidf_ranks[doc_id]:>8} {bm25_ranks[doc_id]:>8} "
              f"{ql_ranks[doc_id]:>10}  {text[:50]}")

compare_models("information retrieval")
print()
compare_models("image learning")

---
## 8. Try Your Own Query
Modify the query below and re-run the cell to see how the models rank the corpus.

In [ ]:
# ✏️ Change this query to anything you want!
my_query = "deep learning image"

compare_models(my_query)

---
## Summary

| Model | Key Idea | Strengths | Weaknesses |
|---|---|---|---|
| **Boolean** | Exact keyword match | Fast, simple | No ranking, rigid |
| **Unigram LM** | Word probability, bag of words | Simple probabilistic foundation | Ignores word order |
| **Bigram LM** | Conditional word probability | Captures local word order | Sparse data, no long-range context |
| **TF-IDF** | Term frequency × rarity | Rewards rare, relevant terms | Doesn't handle doc length well |
| **BM25** | TF-IDF with saturation & length norm | Industry standard, robust | Still bag-of-words |
| **Query Likelihood** | P(query \| doc model) | Probabilistic, principled | Needs smoothing for unseen terms |
